# PPO Training Curves Plotter

This notebook reads training metadata (`model_info.json`) generated by `experiments/train_ppo.py` under the `ray_results/` directory and plots training rewards over iterations for all trials.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

def plot_grid_search_curves(ray_results_dir="ray_results"):
    """
    Discovers all model_info.json metadata files inside ray_results/ and plots
    training rewards over learning iterations for all trials.
    """
    results_path = Path(ray_results_dir).resolve()
    all_runs = []
    
    # Recursively locate all model_info.json files
    for model_info_file in results_path.glob("**/model_info.json"):
        try:
            with open(model_info_file, 'r') as f:
                data = json.load(f)
                
            config_name = data.get('config_name', model_info_file.parent.name)
            learning_curves = data.get('learning_curves', [])
            
            # Extract reward data points
            for point in learning_curves:
                all_runs.append({
                    'Config': config_name,
                    'Iteration': point['iteration'],
                    'Mean Reward': point['reward_mean']
                })
        except Exception as e:
            print(f"Error reading {model_info_file.name}: {e}")
            
    if not all_runs:
        print(f"No model_info.json files or training learning curves found under: {results_path}")
        return
        
    df = pd.DataFrame(all_runs)
    
    # Generate plot
    plt.figure()
    sns.lineplot(data=df, x='Iteration', y='Mean Reward', hue='Config', marker="o")
    plt.title("PPO Split-Policy Multi-Agent Training Curves (MLP DNN)", fontsize=14, fontweight='bold')
    plt.xlabel("Training Iteration", fontsize=12)
    plt.ylabel("Episode Mean Reward (Scaled)", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Discover and plot training curves
plot_grid_search_curves("ray_results")